# **Vanilla GAN Implementation IN KERAS**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.datasets import mnist
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,LeakyReLU,Reshape,Flatten
from tensorflow.keras.optimizers import Adam
import tensorflow as tf

In [2]:
# SET RANDOM SEED
np.random.seed(42)
tf.random.set_seed(42)

In [3]:
# LOAD AND NORMALIZE MNIST DATASET
(x_train,_),(_,_) = mnist.load_data()
x_train = x_train / 127.5 -10
x_train = x_train.reshape(-1,28*28)

11490434/11490434 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [5]:
# Set Dimensions
img_shape = (28,28)
latent_dim = 100

In [19]:
# -----------------------
# Build Generator
# -----------------------
def build_generator():
  model = Sequential([
      tf.keras.Input(shape=(latent_dim,)),
      Dense(128),
      LeakyReLU(0.2),
      Dense(784,activation='tanh'),
      Reshape(img_shape)
  ])
  return model

In [20]:
# -----------------------
# Build Discriminator
# -----------------------
def build_discriminator():
  model = Sequential([
      tf.keras.Input(shape=img_shape),
      Flatten(),
      Dense(128),
      LeakyReLU(0.2),
      Dense(1,activation='sigmoid')
  ])
  return model

In [16]:
# Build and compile Discriminator
discriminator = build_discriminator()
discriminator.compile(loss='binary_crossentropy',optimizer=Adam(0.0002,0.5))

# Build Generator
generator = build_generator()

# Create GAN model (Generator+Discriminator)
z = tf.keras.Input(shape=(latent_dim,))
img = generator(z)
discriminator.trainable = False
validity = discriminator(img)
gan = tf.keras.Model(z,validity)
gan.compile(loss='binary_crossentropy',optimizer=Adam(0.0002,0.5))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/reshaping/flatten.py:37: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)
/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [22]:
from __future__ import generator_stop
def train(epochs,batch_size=128,sample_interval=1000):
  half_batch = batch_size // 2

  for epoch in range(epochs):

    # Train Discriminator
    idx = np.random.randint(0,x_train.shape[0],half_batch)
    real_imgs = x_train[idx].reshape(-1,28,28)

    noise = np.random.normal(0,1,(half_batch,latent_dim))
    fake_imgs = generator.predict(noise)

    # Labels for real and fake images
    real_labels = np.ones((half_batch, 1))
    fake_labels = np.zeros((half_batch, 1))

    d_loss_real = discriminator.train_on_batch(real_imgs, real_labels)
    d_loss_fake = discriminator.train_on_batch(fake_imgs, fake_labels)
    d_loss = 0.5 * np.add(d_loss_real,d_loss_fake)

    # Train Generator
    noise = np.random.normal(0,1,(batch_size,latent_dim))
    g_loss = gan.train_on_batch(noise,np.ones((batch_size,1)))

    # Display progress
    if epoch % sample_interval == 0:
      print(f"{epoch} [D loss: {d_loss:.4f}] [G loss: {g_loss:.4f}]")
      sample_images(epoch)


# ----------------------------
# Image Sampling
# ----------------------------
def sample_images(epoch,n=5):
  noise = np.random.normal(0,1,(n*n,latent_dim))
  gen_imgs = generator.predict(noise)

  gen_imgs = 0.5 * gen_imgs + 0.5

  fig,axs = plt.subplots(n,n)
  cnt = 0
  for i in range(n):
    for j in range(n):
      axs[i,j].imshow(gen_imgs[cnt],cmap='gray')
      axs[i,j].axis('off')
      cnt += 1
  plt.tight_layout()
  plt.savefig(f"gan_mnist_epoch_{epoch}.png")
  plt.show()
  plt.close()

In [23]:
train(epochs=5001, batch_size=64, sample_interval=500)

Output hidden; open in https://colab.research.google.com to view.